#### **ACTIVE SUBSPACES INTERACTIVE EXPLORER**


In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import sys
from pathlib import Path

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "boed" / "__init__.py").is_file():
            return p
        if (p / "pyBOED" / "boed" / "__init__.py").is_file():
            return p / "pyBOED"
    return cwd

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for name in list(sys.modules):
    if name == "boed" or name.startswith("boed."):
        del sys.modules[name]

repo_root = PROJECT_ROOT
print(f"Using PROJECT_ROOT={PROJECT_ROOT}")
from boed.reduction.methods import ActiveSubspaces, generate_ridge_function

Using PROJECT_ROOT=/home/mdoumbou/Documents/Biblio_thèse/pyBOED


In [2]:
# ============================================================================
# WIDGETS
# ============================================================================

# Paramètres de la fonction
d_input_slider = widgets.IntSlider(
    value=20,
    min=5,
    max=100,
    step=5,
    description='Input dim (d):',
    continuous_update=False
)

r_true_slider = widgets.IntSlider(
    value=2,
    min=1,
    max=10,
    step=1,
    description='True AS dim (r):',
    continuous_update=False
)

noise_as_slider = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Noise level:',
    continuous_update=False
)

# Paramètres d'échantillonnage
n_samples_as_slider = widgets.IntSlider(
    value=500,
    min=100,
    max=5000,
    step=100,
    description='# Samples:',
    continuous_update=False
)

# Paramètres AS
activity_threshold_slider = widgets.FloatSlider(
    value=0.99,
    min=0.90,
    max=0.9999,
    step=0.01,
    description='Activity %:',
    continuous_update=False
)

# Distribution des inputs
input_dist_dropdown = widgets.Dropdown(
    options=[
        ('Standard Normal N(0,I)', 'standard'),
        ('Uniform [-1,1]', 'uniform'),
        ('Correlated Normal', 'correlated')
    ],
    value='standard',
    description='Input dist:'
)

compute_button_as = widgets.Button(
    description='Compute AS',
    button_style='success',
    icon='play'
)

output_as = widgets.Output()

In [3]:
# ============================================================================
# FONCTIONS
# ============================================================================

def sample_inputs(n_samples, d, dist_type):
    """Échantillonne les inputs selon la distribution."""
    
    if dist_type == 'standard':
        return np.random.randn(n_samples, d)
    
    elif dist_type == 'uniform':
        return np.random.uniform(-1, 1, (n_samples, d))
    
    elif dist_type == 'correlated':
        # Covariance avec décroissance
        Sigma = np.zeros((d, d))
        for i in range(d):
            for j in range(d):
                Sigma[i, j] = np.exp(-abs(i - j) / 5.0)
        
        return np.random.multivariate_normal(np.zeros(d), Sigma, n_samples)


def compute_as_interactive(button=None):
    """Calcule AS et visualise."""
    
    with output_as:
        clear_output(wait=True)
        
        print("⏳ Génération de la fonction ridge...")
        
        # Générer fonction
        G_func, true_directions = generate_ridge_function(
            d=d_input_slider.value,
            r=r_true_slider.value,
            noise_level=noise_as_slider.value,
            random_state=42
        )
        
        print(f"✓ Fonction G: ℝ^{d_input_slider.value} → ℝ")
        print(f"  Dimension active vraie: {r_true_slider.value}")
        
        print(f"\n⏳ Échantillonnage ({input_dist_dropdown.label})...")
        
        # Échantillonnage
        X_samples = sample_inputs(
            n_samples_as_slider.value,
            d_input_slider.value,
            input_dist_dropdown.value
        )
        
        print(f"✓ {n_samples_as_slider.value} échantillons générés")
        
        print("\n⏳ Calcul des gradients...")
        
        # Évaluation
        G_values = np.zeros(n_samples_as_slider.value)
        gradients = np.zeros((n_samples_as_slider.value, d_input_slider.value))
        
        for i in range(n_samples_as_slider.value):
            G_values[i], gradients[i] = G_func(X_samples[i], return_gradient=True)
        
        print("✓ Gradients calculés")
        
        print("\n⏳ Construction Active Subspace...")
        
        # AS
        as_model = ActiveSubspaces(activity_threshold=activity_threshold_slider.value)
        as_model.fit(gradients)
        
        print(f"✓ AS calculé: {as_model.n_components} directions actives")
        
        # ====================================================================
        # VISUALISATION
        # ====================================================================
        
        fig = make_subplots(
            rows=2, cols=3,
            subplot_titles=(
                'Activity Plot (Eigenvalues)',
                'Cumulative Activity',
                'Sufficient Summary Plot (1D)',
                'Subspace Distance',
                'Shadow Plot (2D)' if as_model.n_components >= 2 else 'N/A',
                'Direction Alignment'
            ),
            specs=[
                [{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}],
                [{'type': 'bar'}, {'type': 'scatter'}, {'type': 'heatmap'}]
            ],
            vertical_spacing=0.15,
            horizontal_spacing=0.12
        )
        
        # SUBPLOT 1 : Activity plot
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(as_model.eigenvalues_) + 1)),
                y=as_model.eigenvalues_,
                mode='markers+lines',
                marker=dict(size=8, color='blue'),
                name='λᵢ'
            ),
            row=1, col=1
        )
        
        fig.add_vline(x=as_model.n_components, line_dash='dash', line_color='red',
                      annotation_text=f'r={as_model.n_components}', row=1, col=1)
        
        fig.update_yaxes(type='log', title_text='Activity λᵢ', row=1, col=1)
        fig.update_xaxes(title_text='Direction', row=1, col=1)
        
        # SUBPLOT 2 : Cumulative
        cumsum = np.cumsum(as_model.activity_ratio_) * 100
        
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(cumsum) + 1)),
                y=cumsum,
                mode='lines',
                fill='tozeroy',
                line=dict(color='green', width=3),
                name='Cumulative'
            ),
            row=1, col=2
        )
        
        fig.add_hline(y=activity_threshold_slider.value * 100,
                      line_dash='dash', line_color='red', row=1, col=2)
        
        fig.update_yaxes(title_text='Activity (%)', row=1, col=2)
        fig.update_xaxes(title_text='Directions', row=1, col=2)
        
        # SUBPLOT 3 : Sufficient summary (1D)
        X_active = as_model.transform(X_samples)
        
        fig.add_trace(
            go.Scatter(
                x=X_active[:, 0],
                y=G_values,
                mode='markers',
                marker=dict(
                    size=5,
                    color=G_values,
                    colorscale='Viridis',
                    showscale=True
                ),
                name='G(x)'
            ),
            row=1, col=3
        )
        
        fig.update_xaxes(title_text='u₁ᵀ x (1st active var)', row=1, col=3)
        fig.update_yaxes(title_text='G(x)', row=1, col=3)
        
        # SUBPLOT 4 : Distance to true subspace
        P_true = true_directions @ true_directions.T
        P_est = as_model.active_directions_ @ as_model.active_directions_.T
        
        dist = np.linalg.norm(P_true - P_est, 'fro')
        
        singular_values = np.linalg.svd(
            true_directions.T @ as_model.active_directions_,
            compute_uv=False
        )
        principal_angle = np.arccos(np.clip(singular_values.min(), 0, 1))
        
        fig.add_trace(
            go.Bar(
                x=['Subspace Distance', 'Principal Angle (deg)'],
                y=[dist, np.degrees(principal_angle)],
                marker_color=['blue', 'orange'],
                text=[f'{dist:.4f}', f'{np.degrees(principal_angle):.2f}°'],
                textposition='auto'
            ),
            row=2, col=1
        )
        
        # SUBPLOT 5 : Shadow plot (2D) si applicable
        if as_model.n_components >= 2:
            fig.add_trace(
                go.Scatter(
                    x=X_active[:, 0],
                    y=X_active[:, 1],
                    mode='markers',
                    marker=dict(
                        size=5,
                        color=G_values,
                        colorscale='Plasma',
                        showscale=True
                    ),
                    name='Shadow'
                ),
                row=2, col=2
            )
            
            fig.update_xaxes(title_text='u₁ᵀ x', row=2, col=2)
            fig.update_yaxes(title_text='u₂ᵀ x', row=2, col=2)
        
        # SUBPLOT 6 : Overlap heatmap
        r_show = min(as_model.n_components, r_true_slider.value, 5)
        overlap = np.abs(true_directions[:, :r_show].T @ 
                        as_model.active_directions_[:, :r_show])
        
        fig.add_trace(
            go.Heatmap(
                z=overlap,
                x=[f'Est {i+1}' for i in range(r_show)],
                y=[f'True {i+1}' for i in range(r_show)],
                colorscale='Blues',
                zmin=0, zmax=1
            ),
            row=2, col=3
        )
        
        # Layout
        fig.update_layout(
            height=900,
            showlegend=True,
            title_text=f'<b>Active Subspaces Analysis</b> (d={d_input_slider.value}, r_true={r_true_slider.value})'
        )
        
        fig.show()
        
        # Stats
        print("\n" + "="*70)
        print("STATISTIQUES")
        print("="*70)
        print(f"Dimension input         : {d_input_slider.value}")
        print(f"Dimension active vraie  : {r_true_slider.value}")
        print(f"Dimension active estimée: {as_model.n_components}")
        print(f"Distance sous-espace    : {dist:.4f}")
        print(f"Angle principal         : {np.degrees(principal_angle):.2f}°")
        print(f"\nPremiers eigenvalues :")
        for i in range(min(10, len(as_model.eigenvalues_))):
            print(f"  λ_{i+1} = {as_model.eigenvalues_[i]:.6e} ({as_model.activity_ratio_[i]*100:.2f}%)")

compute_button_as.on_click(compute_as_interactive)

In [4]:
# ============================================================================
# INTERFACE
# ============================================================================

function_params = widgets.VBox([
    widgets.HTML("<h3>🎯 Ridge Function</h3>"),
    d_input_slider,
    r_true_slider,
    noise_as_slider
])

sampling_params = widgets.VBox([
    widgets.HTML("<h3>📊 Sampling</h3>"),
    n_samples_as_slider,
    input_dist_dropdown
])

as_params = widgets.VBox([
    widgets.HTML("<h3>🔧 Active Subspace</h3>"),
    activity_threshold_slider,
    compute_button_as
])

controls_as = widgets.HBox([function_params, sampling_params, as_params])

display(widgets.VBox([
    widgets.HTML("<h1>🎲 Active Subspaces Interactive Explorer</h1>"),
    controls_as,
    output_as
]))

compute_as_interactive()